# 04 - Overpass-level block bootstrap for SWOT-HF-radar validation

This notebook estimates uncertainty while preserving dependence among grid-cell pairs from the same SWOT overpass. The 33 complete native, coarsened-SWOT, and HF-radar triplets are treated as the resampling blocks.

No figures are saved or generated here. The output is a readable table of pooled and overpass-balanced estimates with 95% percentile bootstrap intervals.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

REPO = Path.cwd().resolve().parent
PROCESSED_DIR = REPO / 'data' / 'processed' / 'swot_hfradar_matches'
COARSE_DIR = PROCESSED_DIR / 'swot_coarsened_masked'
RADAR_DIR = PROCESSED_DIR / 'hfradar_swot_masked'
MIN_DIRECTION_SPEED = 0.05
BOOTSTRAP_REPLICATES = 5000
RANDOM_SEED = 42


def indexed_files(directory, prefix):
    indexed = {}
    for path in sorted(directory.glob(f'{prefix}_*.nc')):
        with xr.open_dataset(path) as dataset:
            if 'cycle' in dataset.attrs and 'pass' in dataset.attrs:
                indexed[(int(dataset.attrs['cycle']), int(dataset.attrs['pass']))] = path
    return indexed

coarse_files = indexed_files(COARSE_DIR, 'swot_coarsened_masked')
radar_files = indexed_files(RADAR_DIR, 'hfradar_swot_masked')
required_events = sorted(set(coarse_files) & set(radar_files))

if not required_events:
    raise FileNotFoundError('No complete coarsened-SWOT/HF-radar pairs were found.')

print(f'Repository: {REPO}')
print(f'Matched products directory: {PROCESSED_DIR}')
print(f'Complete overpasses available: {len(required_events)}')
print(f'Bootstrap replicates: {BOOTSTRAP_REPLICATES:,}')
print(f'Random seed: {RANDOM_SEED}')

Repository: <repo>
Matched products directory: <repo>/data/processed/swot_hfradar_matches
Complete overpasses available: 33
Bootstrap replicates: 5,000
Random seed: 42


In [2]:
def wrapped_direction_error(model_direction, reference_direction):
    return (model_direction - reference_direction + 180.0) % 360.0 - 180.0


def load_overpass_record(event_id):
    with xr.open_dataset(coarse_files[event_id]) as coarse_dataset:
        coarse_u = coarse_dataset['ugos_coarsened_raw'].values.astype(float)
        coarse_v = coarse_dataset['vgos_coarsened_raw'].values.astype(float)
        comparison_mask = coarse_dataset['comparison_mask'].values.astype(bool)
        swot_time = pd.to_datetime(coarse_dataset.attrs.get('swot_time', ''))

    with xr.open_dataset(radar_files[event_id]) as radar_dataset:
        radar_u = radar_dataset['ewct_raw'].values.astype(float)
        radar_v = radar_dataset['nsct_raw'].values.astype(float)

    coarse_speed = np.hypot(coarse_u, coarse_v)
    radar_speed = np.hypot(radar_u, radar_v)
    valid = (
        comparison_mask
        & np.isfinite(coarse_u)
        & np.isfinite(coarse_v)
        & np.isfinite(radar_u)
        & np.isfinite(radar_v)
        & (coarse_speed >= MIN_DIRECTION_SPEED)
        & (radar_speed >= MIN_DIRECTION_SPEED)
    )

    coarse_speed = coarse_speed[valid]
    radar_speed = radar_speed[valid]
    coarse_direction = (np.degrees(np.arctan2(coarse_v[valid], coarse_u[valid])) + 360.0) % 360.0
    radar_direction = (np.degrees(np.arctan2(radar_v[valid], radar_u[valid])) + 360.0) % 360.0
    direction_error = wrapped_direction_error(coarse_direction, radar_direction)
    vector_error = np.hypot(coarse_u[valid] - radar_u[valid], coarse_v[valid] - radar_v[valid])

    return {
        'event_id': event_id,
        'time': swot_time,
        'coarse_speed': coarse_speed,
        'radar_speed': radar_speed,
        'speed_error': coarse_speed - radar_speed,
        'reference_speed': (coarse_speed + radar_speed) / 2,
        'vector_error': vector_error,
        'direction_error': direction_error,
    }


overpass_records = [load_overpass_record(event_id) for event_id in required_events]
print(f'Loaded overpass records: {len(overpass_records)}')
print(f'Valid paired observations: {sum(record["speed_error"].size for record in overpass_records):,}')

Loaded overpass records: 33
Valid paired observations: 6,216


In [3]:
def circular_mean_degrees(errors):
    error_radians = np.radians(errors)
    return float(np.degrees(np.arctan2(
        np.mean(np.sin(error_radians)),
        np.mean(np.cos(error_radians)),
    )))


def pooled_metrics(records):
    speed_errors = np.concatenate([record['speed_error'] for record in records])
    vector_errors = np.concatenate([record['vector_error'] for record in records])
    direction_errors = np.concatenate([record['direction_error'] for record in records])

    return {
        'speed_bias_mps': float(np.mean(speed_errors)),
        'speed_mae_mps': float(np.mean(np.abs(speed_errors))),
        'speed_rmse_mps': float(np.sqrt(np.mean(speed_errors ** 2))),
        'vector_mae_mps': float(np.mean(vector_errors)),
        'vector_rmse_mps': float(np.sqrt(np.mean(vector_errors ** 2))),
        'direction_circular_bias_deg': circular_mean_degrees(direction_errors),
        'direction_mae_deg': float(np.mean(np.abs(direction_errors))),
        'direction_rmse_deg': float(np.sqrt(np.mean(direction_errors ** 2))),
        'direction_within_15_pct': float(100 * np.mean(np.abs(direction_errors) <= 15)),
        'direction_within_30_pct': float(100 * np.mean(np.abs(direction_errors) <= 30)),
        'direction_within_45_pct': float(100 * np.mean(np.abs(direction_errors) <= 45)),
    }


def balanced_metrics(records):
    event_metric_rows = [pooled_metrics([record]) for record in records]
    return {
        metric_name: float(np.mean([row[metric_name] for row in event_metric_rows]))
        for metric_name in event_metric_rows[0]
    }


pooled_estimate = pooled_metrics(overpass_records)
balanced_estimate = balanced_metrics(overpass_records)

print('Pooled estimates:')
print(pd.Series(pooled_estimate).round(4).to_string())
print('\nOverpass-balanced estimates:')
print(pd.Series(balanced_estimate).round(4).to_string())

Pooled estimates:
speed_bias_mps                 -0.0970
speed_mae_mps                   0.2788
speed_rmse_mps                  0.3868
vector_mae_mps                  0.4075
vector_rmse_mps                 0.5040
direction_circular_bias_deg     3.3713
direction_mae_deg              35.7983
direction_rmse_deg             55.4620
direction_within_15_pct        45.1577
direction_within_30_pct        63.9157
direction_within_45_pct        73.7773

Overpass-balanced estimates:
speed_bias_mps                 -0.1175
speed_mae_mps                   0.2765
speed_rmse_mps                  0.3560
vector_mae_mps                  0.3968
vector_rmse_mps                 0.4629
direction_circular_bias_deg     5.1133
direction_mae_deg              37.9082
direction_rmse_deg             51.0499
direction_within_15_pct        46.6102
direction_within_30_pct        63.3467
direction_within_45_pct        71.9435


In [4]:
rng = np.random.default_rng(RANDOM_SEED)
overpass_count = len(overpass_records)
bootstrap_indices = rng.integers(
    0,
    overpass_count,
    size=(BOOTSTRAP_REPLICATES, overpass_count),
)
metric_names = list(pooled_estimate)
pooled_bootstrap = {metric_name: np.empty(BOOTSTRAP_REPLICATES) for metric_name in metric_names}
balanced_bootstrap = {metric_name: np.empty(BOOTSTRAP_REPLICATES) for metric_name in metric_names}

for replicate_index, sampled_indices in enumerate(bootstrap_indices):
    sampled_records = [overpass_records[index] for index in sampled_indices]
    pooled_result = pooled_metrics(sampled_records)
    balanced_result = balanced_metrics(sampled_records)
    for metric_name in metric_names:
        pooled_bootstrap[metric_name][replicate_index] = pooled_result[metric_name]
        balanced_bootstrap[metric_name][replicate_index] = balanced_result[metric_name]

print(f'Completed overpass-block bootstrap replicates: {BOOTSTRAP_REPLICATES:,}')

Completed overpass-block bootstrap replicates: 5,000


In [5]:
summary_rows = []
for metric_name in metric_names:
    metric_label = metric_name.replace('_', ' ')
    pooled_lower, pooled_upper = np.percentile(
        pooled_bootstrap[metric_name],
        [2.5, 97.5],
    )
    balanced_lower, balanced_upper = np.percentile(
        balanced_bootstrap[metric_name],
        [2.5, 97.5],
    )
    summary_rows.extend([
        {
            'metric': metric_label,
            'estimator': 'Pooled pairs',
            'estimate': pooled_estimate[metric_name],
            'lower_95_pct': pooled_lower,
            'upper_95_pct': pooled_upper,
        },
        {
            'metric': metric_label,
            'estimator': 'Overpass balanced',
            'estimate': balanced_estimate[metric_name],
            'lower_95_pct': balanced_lower,
            'upper_95_pct': balanced_upper,
        },
    ])

bootstrap_summary_table = pd.DataFrame(summary_rows)
print(bootstrap_summary_table.round(4).to_string(index=False))

                     metric         estimator  estimate  lower_95_pct  upper_95_pct
             speed bias mps      Pooled pairs   -0.0970       -0.1475       -0.0468
             speed bias mps Overpass balanced   -0.1175       -0.1706       -0.0669
              speed mae mps      Pooled pairs    0.2788        0.2507        0.3079
              speed mae mps Overpass balanced    0.2765        0.2442        0.3094
             speed rmse mps      Pooled pairs    0.3868        0.3413        0.4377
             speed rmse mps Overpass balanced    0.3560        0.3180        0.3964
             vector mae mps      Pooled pairs    0.4075        0.3699        0.4453
             vector mae mps Overpass balanced    0.3968        0.3642        0.4311
            vector rmse mps      Pooled pairs    0.5040        0.4490        0.5577
            vector rmse mps Overpass balanced    0.4629        0.4226        0.5075
direction circular bias deg      Pooled pairs    3.3713       -0.5043       

**The bootstrap asks a simple question: “If we repeated this study with a different set of similar overpasses, how much might the results change?”**

## A1. What the notebook did

Imagine the 33 overpasses are 33 bags of observations:

```text
Overpass 1: many grid-cell comparisons
Overpass 2: many grid-cell comparisons
...
Overpass 33: many grid-cell comparisons
```

The notebook:

1. Randomly selected 33 overpasses.
2. Allowed an overpass to be selected more than once.
3. Kept all grid-cell observations belonging to each selected overpass.
4. Recalculated every metric.
5. Repeated this 5,000 times.

This is called **block bootstrapping**. The whole overpass is kept together because its grid cells are related.

The bootstrap distribution shows how much the result changes when the available overpasses change.

## A2. Why there are two estimates

The notebook reports two ways of averaging.

### Pooled pairs

All 6,216 grid-cell pairs are combined:

```text
overpasses with more valid cells have more influence
```

### Overpass balanced

Each overpass gets equal weight:

```text
calculate each overpass result
then average the 33 overpass results
```

This is often more appropriate when the scientific unit is the overpass rather than the individual grid cell.

## A3. Speed results

| Metric | Pooled estimate | 95% interval | Overpass-balanced estimate | 95% interval |
|---|---:|---:|---:|---:|
| Bias | $-0.097$ m/s | $-0.148$ to $-0.047$ | $-0.118$ m/s | $-0.171$ to $-0.067$ |
| MAE | $0.279$ m/s | $0.251$ to $0.308$ | $0.277$ m/s | $0.244$ to $0.309$ |
| RMSE | $0.387$ m/s | $0.341$ to $0.438$ | $0.356$ m/s | $0.318$ to $0.396$ |
| Vector MAE | $0.408$ m/s | $0.370$ to $0.445$ | $0.397$ m/s | $0.364$ to $0.431$ |
| Vector RMSE | $0.504$ m/s | $0.449$ to $0.558$ | $0.463$ m/s | $0.423$ to $0.508$ |

The speed bias is consistently negative:

> SWOT tends to estimate speeds approximately $0.10$ to $0.12$ m/s slower than HF radar.

The confidence intervals do not include zero, so this negative bias is reasonably stable across overpass resamples.

## A4. Direction results

| Metric | Pooled estimate | 95% interval | Overpass-balanced estimate | 95% interval |
|---|---:|---:|---:|---:|
| Circular bias | $3.37^\circ$ | $-0.50^\circ$ to $7.26^\circ$ | $5.11^\circ$ | $-7.51^\circ$ to $17.84^\circ$ |
| MAE | $35.8^\circ$ | $30.9^\circ$ to $41.1^\circ$ | $37.9^\circ$ | $29.1^\circ$ to $47.8^\circ$ |
| RMSE | $55.5^\circ$ | $49.2^\circ$ to $61.7^\circ$ | $51.0^\circ$ | $41.0^\circ$ to $61.6^\circ$ |
| Within $15^\circ$ | $45.2\%$ | $39.6$ to $50.9% | $46.6\%$ | $37.2$ to $56.3% |
| Within $30^\circ$ | $63.9\%$ | $58.3$ to $69.5% | $63.3\%$ | $53.8$ to $72.2% |
| Within $45^\circ$ | $73.8\%$ | $68.9$ to $78.5% | $71.9\%$ | $62.9$ to $80.1% |

The circular bias is small, but its interval includes zero. Therefore:

> There is no strong evidence of a consistent directional offset, although individual directional errors are large.

The more important result is the spread:

- Typical absolute directional error: roughly $36$–$38^\circ$.
- About 64% of pairs agree within $30^\circ$.
- About 72%–74% agree within $45^\circ$.

## A5. What the bootstrap tells us

The results are not a single fragile number. They are ranges showing how the estimates vary when different combinations of overpasses are considered.

```text
Speed bias:
stable negative offset

Directional bias:
small, but uncertain

Directional agreement:
moderate, with substantial individual disagreement

Vector error:
larger than speed error because it includes direction as well as magnitude
```

## A6. Main conclusion

The validation is strongest for the speed bias:

> Coarsened SWOT is consistently slower than HF radar by about $0.1$ m/s.

Directional agreement is less strong:

> The average directional bias is small, but individual direction errors are substantial, with an RMSE of approximately $51$–$55^\circ$.

The pooled and overpass-balanced results are broadly similar, which suggests that the conclusions are not being driven entirely by a few overpasses with unusually many valid grid cells. Nevertheless, the confidence intervals are based on only 33 overpass-level blocks, so they should be interpreted as uncertainty across this specific set of overpasses, not as universal uncertainty for all conditions.

## How the overpass resampling works

**Each bootstrap replicate draws 33 times from the same set of 33 overpasses, with replacement.**

“With replacement” means that after selecting an overpass, it is put back into the pool and can be selected again.

Example with five overpasses:

```text
Original:    A B C D E

Bootstrap:   A A C D D
```

Here:

- A and D appear twice.
- B and E are omitted.
- The replicate still contains five overpass selections.

The notebook does the same with 33 overpasses:

```text
Original:    33 overpasses
One replicate: 33 selections from those 33, allowing repeats
```

It then calculates the metric using all grid-cell pairs belonging to the selected overpasses. Overpasses selected twice contribute twice; omitted overpasses contribute nothing in that replicate.

After 5,000 such replicates, the spread of the results estimates how much the metrics would vary if a different collection of similar overpasses had been observed.

This does not create new physical observations. It reweights the existing 33 overpasses to estimate uncertainty.